# Robustness to image quality: measured on trained weights

Builds the "degradation -> mAP50" table for an already trained model, **without retraining**: about 10 minutes.

**Notebook settings:** Accelerator: GPU (one is enough); Internet: On.

**Add Input, two inputs:**
1. the MAR20 dataset (`military aircraft recognition`);
2. the model weights: Your Work -> the training notebook -> its Output (where the `*.pt` is).

Weights are found in `/kaggle/input` automatically. The result is `robustness.md` and `robustness.json`
in this version's Output.

In [ ]:
REPO_URL = "https://github.com/miss-mississippi/muxxed_defence_tech.git"
WEIGHTS = None   # None: use the first *.pt in /kaggle/input; otherwise a full path
LIMIT, IMGSZ = 400, 800

In [ ]:
!rm -rf /kaggle/working/repo && git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q ultralytics==8.4.152

In [ ]:
import glob
from pathlib import Path

hits = sorted(Path("/kaggle/input").glob("**/Annotations/Oriented Bounding Boxes"))
if not hits:
    raise SystemExit("MAR20 not found in /kaggle/input: Add Input -> the MAR20 dataset")
MAR20_SRC = str(hits[0].parent.parent)

if WEIGHTS is None:
    found = sorted(Path("/kaggle/input").glob("**/*.pt"))
    if not found:
        raise SystemExit("weights not found in /kaggle/input: Add Input -> Your Work -> Output of the training notebook")
    WEIGHTS = str(found[0])
print("MAR20:", MAR20_SRC, "\nweights:", WEIGHTS)

!python scripts/prepare_mar20.py --src "$MAR20_SRC" --out /kaggle/working/mar20_yolo | tail -2

In [ ]:
!python scripts/robustness.py --data /kaggle/working/mar20_yolo/mar20.yaml \
    --weights "$WEIGHTS" --split test --limit $LIMIT --imgsz $IMGSZ --device 0

In [ ]:
import shutil
from IPython.display import Markdown, display

export = Path("/kaggle/working/export")
export.mkdir(exist_ok=True)
for f in glob.glob("outputs/robustness/robustness.*"):
    shutil.copy2(f, export)
display(Markdown(Path("outputs/robustness/robustness.md").read_text()))
print(sorted(p.name for p in export.iterdir()))